# Etap 2: ETL – czyszczenie i transformacja danych

Dane wejściowe: `data/raw/ceccarelli_clinical_clean.csv`  
Dane wyjściowe: `data/processed/` (pliki gotowe do załadowania do SQLite)

Kroki:
1. Wczytanie i inspekcja surowych danych
2. Analiza braków danych
3. Czyszczenie i ujednolicenie wartości
4. Tworzenie zmiennych pochodnych
5. Eksport do `data/processed/`

In [1]:
import pandas as pd
from pathlib import Path

# Ścieżki
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

# Wczytanie danych
df = pd.read_csv(RAW_DIR / "ceccarelli_clinical_clean.csv")

print(f"Rozmiar: {df.shape}") 
print(f"\nKolumny:\n{df.dtypes}")
print(f"\nPierwsze wiersze:")
df.head()

Rozmiar: (1122, 16)

Kolumny:
patient_id               object
study                    object
histology                object
grade                    object
age_at_diagnosis        float64
gender                   object
kps                     float64
mutation_count          float64
os_months               float64
os_event                float64
idh_status               object
mgmt_status              object
codel_1p19q              object
idh_codel_subtype        object
tert_promoter_status     object
atrx_status              object
dtype: object

Pierwsze wiersze:


,patient_id,study,histology,grade,age_at_diagnosis,gender,kps,mutation_count,os_months,os_event,idh_status,mgmt_status,codel_1p19q,idh_codel_subtype,tert_promoter_status,atrx_status
0,TCGA-CS-4938,Brain Lower Grade Glioma,astrocytoma,G2,31.0,female,90.0,15.0,4.698251,0.0,Mutant,Unmethylated,non-codel,IDHmut-non-codel,WT,Mutant
1,TCGA-CS-4941,Brain Lower Grade Glioma,astrocytoma,G3,67.0,male,90.0,50.0,7.688047,1.0,WT,Methylated,non-codel,IDHwt,Mutant,WT
2,TCGA-CS-4942,Brain Lower Grade Glioma,astrocytoma,G3,44.0,female,90.0,24.0,43.861292,1.0,Mutant,Unmethylated,non-codel,IDHmut-non-codel,WT,Mutant
3,TCGA-CS-4943,Brain Lower Grade Glioma,astrocytoma,G3,37.0,male,50.0,30.0,18.135905,0.0,Mutant,Methylated,non-codel,IDHmut-non-codel,WT,Mutant
4,TCGA-CS-4944,Brain Lower Grade Glioma,astrocytoma,G2,50.0,male,90.0,20.0,10.612133,0.0,Mutant,Methylated,non-codel,IDHmut-non-codel,Mutant,WT


In [2]:
# Analiza braków danych
print("=== Braki danych ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

missing_report = pd.DataFrame({
    'brakujące': missing,
    'procent': missing_pct
})

print(missing_report[missing_report['brakujące'] > 0].to_string())
print(f"\nPacjenci bez ŻADNYCH braków: {df.dropna().shape[0]} / {len(df)}")

=== Braki danych ===
                      brakujące  procent
histology                    75      6.7
grade                        75      6.7
age_at_diagnosis             75      6.7
gender                       75      6.7
kps                         425     37.9
mutation_count              302     26.9
os_months                    75      6.7
os_event                     76      6.8
idh_status                  127     11.3
mgmt_status                 190     16.9
codel_1p19q                  29      2.6
idh_codel_subtype           148     13.2
tert_promoter_status        793     70.7
atrx_status                 302     26.9

Pacjenci bez ŻADNYCH braków: 155 / 1122


In [3]:
# ── Krok 1: Wykluczenie pacjentów bez danych klinicznych ──────────────

# Pacjenci bez os_months nie mają żadnych użytecznych danych klinicznych
# (brakuje im też histologii, wieku, płci itd.)
df_clean = df.dropna(subset=['os_months']).copy()

print(f"Przed: {len(df)} pacjentów")
print(f"Po wykluczeniu bez os_months: {len(df_clean)} pacjentów")
print(f"Wykluczono: {len(df) - len(df_clean)} pacjentów")

Przed: 1122 pacjentów
Po wykluczeniu bez os_months: 1047 pacjentów
Wykluczono: 75 pacjentów


In [4]:
# ── Krok 2: Sprawdzenie unikalnych wartości w kolumnach kategorycznych ──


cat_columns = ['study', 'histology', 'grade', 'gender', 
               'idh_status', 'mgmt_status', 'codel_1p19q', 
               'idh_codel_subtype']

for col in cat_columns:
    unique_vals = df_clean[col].dropna().unique()
    print(f"\n{col} ({len(unique_vals)} unikalnych):")
    print(sorted(unique_vals))
    


study (2 unikalnych):
['Brain Lower Grade Glioma', 'Glioblastoma multiforme']

histology (4 unikalnych):
['astrocytoma', 'glioblastoma', 'oligoastrocytoma', 'oligodendroglioma']

grade (3 unikalnych):
['G2', 'G3', 'G4']

gender (2 unikalnych):
['female', 'male']

idh_status (2 unikalnych):
['Mutant', 'WT']

mgmt_status (2 unikalnych):
['Methylated', 'Unmethylated']

codel_1p19q (2 unikalnych):
['codel', 'non-codel']

idh_codel_subtype (3 unikalnych):
['IDHmut-codel', 'IDHmut-non-codel', 'IDHwt']


In [5]:
# ── Krok 3: Ujednolicenie wartości ────────────────────────────────────

# Skracamy nazwy badań do standardowych kodów TCGA
df_clean['study'] = df_clean['study'].map({
    'Brain Lower Grade Glioma': 'LGG',
    'Glioblastoma multiforme': 'GBM'
})

print(df_clean['study'].value_counts())

study
GBM    590
LGG    457
Name: count, dtype: int64


In [6]:
# ── Krok 4: Naprawa typów danych ──────────────────────────────────────

df_clean['os_event'] = df_clean['os_event'].astype('Int64')

print(df_clean[['os_months', 'os_event']].dtypes)
print(f"\nUnikalne wartości os_event: {sorted(df_clean['os_event'].dropna().unique())}")

os_months    float64
os_event       Int64
dtype: object

Unikalne wartości os_event: [np.int64(0), np.int64(1)]


In [7]:
# ── Krok 5: Zmienne pochodne (binarne 0/1) ────────────────────────────

import numpy as np

# IDH: Mutant = 1, WT = 0
df_clean['idh_mutant'] = pd.array(
    np.where(df_clean['idh_status'] == 'Mutant', 1,
    np.where(df_clean['idh_status'].isna(), pd.NA, 0)),
    dtype='Int64'
)

# MGMT: Methylated = 1, Unmethylated = 0
df_clean['mgmt_methylated'] = pd.array(
    np.where(df_clean['mgmt_status'] == 'Methylated', 1,
    np.where(df_clean['mgmt_status'].isna(), pd.NA, 0)),
    dtype='Int64'
)

# 1p/19q codeletion: codel = 1, non-codel = 0
df_clean['codel'] = pd.array(
    np.where(df_clean['codel_1p19q'] == 'codel', 1,
    np.where(df_clean['codel_1p19q'].isna(), pd.NA, 0)),
    dtype='Int64'
)

# Sprawdzenie
print(df_clean[['idh_status', 'idh_mutant', 
                'mgmt_status', 'mgmt_methylated',
                'codel_1p19q', 'codel']].head(8))

  idh_status  idh_mutant   mgmt_status  mgmt_methylated codel_1p19q  codel
0     Mutant           1  Unmethylated                0   non-codel      0
1         WT           0    Methylated                1   non-codel      0
2     Mutant           1  Unmethylated                0   non-codel      0
3     Mutant           1    Methylated                1   non-codel      0
4     Mutant           1    Methylated                1   non-codel      0
5     Mutant           1    Methylated                1       codel      1
6     Mutant           1    Methylated                1   non-codel      0
7     Mutant           1    Methylated                1   non-codel      0


In [8]:
# ── Krok 6: Eksport do data/processed/ ───────────────────────────────

output_path = PROCESSED_DIR / "clinical_processed.csv"
df_clean.to_csv(output_path, index=False)

print(f"Zapisano: {output_path}")
print(f"Rozmiar: {df_clean.shape}")
print(f"\nKolumny końcowe:\n{df_clean.dtypes}")

Zapisano: ..\data\processed\clinical_processed.csv
Rozmiar: (1047, 19)

Kolumny końcowe:
patient_id               object
study                    object
histology                object
grade                    object
age_at_diagnosis        float64
gender                   object
kps                     float64
mutation_count          float64
os_months               float64
os_event                  Int64
idh_status               object
mgmt_status              object
codel_1p19q              object
idh_codel_subtype        object
tert_promoter_status     object
atrx_status              object
idh_mutant                Int64
mgmt_methylated           Int64
codel                     Int64
dtype: object


## Raport z czyszczenia danych

**Dane wejściowe:** `data/raw/ceccarelli_clinical_clean.csv` (1122 pacjentów, 16 kolumn)  
**Dane wyjściowe:** `data/processed/clinical_processed.csv` (1047 pacjentów, 19 kolumn)

### Decyzje podjęte podczas czyszczenia

| Krok | Decyzja | Uzasadnienie |
|------|---------|--------------|
| Wykluczenie 75 pacjentów | Usunięto wiersze bez `os_months` | Brak jakichkolwiek danych klinicznych – nie nadają się do analizy przeżycia |
| `study` | Zamieniono pełne nazwy na kody: LGG / GBM | Standardowe kody TCGA, wygodniejsze w SQL |
| `os_event` | Zmieniono typ float64 → Int64 | Zmienna binarna (0/1) nie powinna być zmiennoprzecinkowa |
| `kps` (37.9% braków) | Pozostawiono bez zmian | Wysoka niekompletność typowa dla TCGA; zmienna pomocnicza, nie wejdzie do głównego modelu |
| `tert_promoter_status` (70.7% braków) | Pozostawiono bez zmian | Zbyt mało danych do analizy; dostępna w bazie, wykluczona z modeli |
| Nowe kolumny binarne | Dodano `idh_mutant`, `mgmt_methylated`, `codel` (0/1/NA) | Wymagane przez model Coxa i zapytania SQL |
| Braki w IDH/MGMT | Pozostawiono jako NA | Pacjenci bez biomarkera nie są wykluczani – pomijani tylko w analizach wymagających tej zmiennej |